# PCEPS Calibration Analysis

Evaluates PCEPS probability calibration and predictive performance.
Produces FIGURE_2 (reliability diagram) and FIGURE_3 (AUC-ROC).

In [ ]:
# Cell 1: Load scores and labels
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_curve, auc, brier_score_loss
import json

DATASET_DIR = Path('research/datasets/phantom-v1')
traces = pd.read_parquet(DATASET_DIR / 'traces.parquet')
labels = pd.read_parquet(DATASET_DIR / 'labels.parquet')

# Use traces with PCEPS scores for pre-compromise windows
pceps_traces = traces[traces['phantom_pceps_score'].notna()].copy()
print(f"Traces with PCEPS scores: {len(pceps_traces):,}")
print(f"Attack traces: {(pceps_traces['label'] == 1).sum():,}")
print(f"Benign traces: {(pceps_traces['label'] == 0).sum():,}")
print(f"Score range: [{pceps_traces['phantom_pceps_score'].min():.3f}, "
      f"{pceps_traces['phantom_pceps_score'].max():.3f}]")


In [ ]:
# Cell 2: Reliability diagram (calibration plot) — FIGURE_2

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

y_true = pceps_traces['label'].values
y_prob = pceps_traces['phantom_pceps_score'].values

# 10 equally spaced probability bins
prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=10, strategy='uniform')

ax = axes[0]
ax.plot(prob_pred, prob_true, 's-', color='#2ecc71', label='PHANTOM (Platt-scaled)', linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration', alpha=0.7)

# Naive baselines (illustrative)
ax.axhline(y=y_true.mean(), color='#e74c3c', linestyle=':', label=f'Always-mean ({y_true.mean():.2f})', alpha=0.8)

ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('FIGURE_2: PCEPS Reliability Diagram\n(10-bin calibration plot)')
ax.legend(loc='upper left')
ax.set_xlim(0, 1)
ax.set_ylim(-0.05, 1.05)
ax.grid(True, alpha=0.3)

# Brier score decomposition (gap histogram)
axes[1].hist(y_prob[y_true == 0], bins=20, alpha=0.6, color='#3498db', label='Benign (y=0)', density=True)
axes[1].hist(y_prob[y_true == 1], bins=20, alpha=0.6, color='#e74c3c', label='Attack (y=1)', density=True)
axes[1].set_xlabel('PCEPS Predicted Probability')
axes[1].set_ylabel('Density')
axes[1].set_title('Score Distribution by True Label')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('research/evaluation/results/figure_2_pceps_reliability.pdf', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Cell 3: Brier score computation

bs_phantom = brier_score_loss(y_true, y_prob)
bs_always_zero = brier_score_loss(y_true, np.zeros_like(y_prob))
bs_always_one  = brier_score_loss(y_true, np.ones_like(y_prob))
bs_prevalence  = brier_score_loss(y_true, np.full_like(y_prob, y_true.mean()))

print("Brier Scores (lower = better calibration):")
print(f"  PHANTOM (Platt-scaled):   {bs_phantom:.4f}")
print(f"  Always predict 0:         {bs_always_zero:.4f}")
print(f"  Always predict 1:         {bs_always_one:.4f}")
print(f"  Always predict prevalence:{bs_prevalence:.4f}")

# Per-attack-family breakdown
for family in pceps_traces['attack_family'].dropna().unique():
    mask = pceps_traces['attack_family'] == family
    if mask.sum() > 5:
        bs = brier_score_loss(
            pceps_traces[mask]['label'].values,
            pceps_traces[mask]['phantom_pceps_score'].values,
        )
        print(f"  {family:<40}: {bs:.4f}")


In [ ]:
# Cell 4: AUC-ROC curve — FIGURE_3

fig, ax = plt.subplots(figsize=(7, 6))

# PHANTOM
fpr_ph, tpr_ph, _ = roc_curve(y_true, y_prob)
roc_auc_ph = auc(fpr_ph, tpr_ph)
ax.plot(fpr_ph, tpr_ph, color='#2ecc71', linewidth=2.5,
        label=f'PHANTOM (AUC = {roc_auc_ph:.3f})')

# KL-score baseline (using kl_score column as a naive ranker)
if 'kl_score' in pceps_traces.columns and pceps_traces['kl_score'].notna().any():
    kl_scores = pceps_traces['kl_score'].fillna(0).values
    fpr_kl, tpr_kl, _ = roc_curve(y_true, kl_scores)
    roc_auc_kl = auc(fpr_kl, tpr_kl)
    ax.plot(fpr_kl, tpr_kl, color='#3498db', linestyle='--', linewidth=2,
            label=f'KL-score only (AUC = {roc_auc_kl:.3f})')

# Random baseline
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random (AUC = 0.500)')

ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('FIGURE_3: AUC-ROC Comparison\n(PHANTOM vs Baselines)')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)

plt.tight_layout()
plt.savefig('research/evaluation/results/figure_3_pceps_auc.pdf', dpi=150, bbox_inches='tight')
plt.show()
print(f"PHANTOM AUC-ROC: {roc_auc_ph:.4f}")


In [ ]:
# Cell 5: Feature importance (XGBoost gain-based, if model available)
from pathlib import Path

model_path = Path('research/datasets/raw/pceps_model.json')
if model_path.exists():
    try:
        import xgboost as xgb
        model = xgb.XGBClassifier()
        model.load_model(str(model_path))

        importance = model.get_booster().get_score(importance_type='gain')
        imp_series = pd.Series(importance).sort_values(ascending=True).tail(20)

        fig, ax = plt.subplots(figsize=(8, 6))
        imp_series.plot(kind='barh', ax=ax, color='#2ecc71')
        ax.set_xlabel('Feature Importance (Gain)')
        ax.set_title('XGBoost PCEPS Feature Importance\n(Top 20 by gain)')
        ax.axvline(x=0, color='black', linewidth=0.5)

        # Highlight causal_effect feature (f1 from DoWhy)
        for tick in ax.get_yticklabels():
            if 'causal_effect' in tick.get_text().lower():
                tick.set_fontweight('bold')
                tick.set_color('#e74c3c')

        plt.tight_layout()
        plt.savefig('research/evaluation/results/figure_pceps_feature_importance.pdf',
                    dpi=150, bbox_inches='tight')
        plt.show()
        print("Key finding: Does causal_effect rank in top features?",
              any('causal_effect' in k.lower() for k in list(importance)[:5]))
    except ImportError:
        print("xgboost not installed; skipping feature importance plot.")
else:
    print(f"Model not found at {model_path}; run PCEPS training first.")
